In [23]:
import pandas as pd
import geopandas as gpd
import json
import matplotlib.pyplot as plt

Import data:

In [24]:
# 1. Load your full dataset
df = pd.read_csv('/Users/lucasselvik/Desktop/ARC Project/california_202008_202009.csv')

# 2. Parse the string list into actual Python lists
df['STOPS_LIST'] = df['STOPS_BY_DAY'].apply(json.loads)

# 3. "Explode" the lists into a Long Format
df_long = df[['AREA', 'STOPS_LIST']].explode('STOPS_LIST')
df_long['Day'] = df_long.groupby('AREA').cumcount() + 1
df_long['Stops'] = df_long['STOPS_LIST'].astype(float)

# THE FIX: Ensure it is a string, and pad it with zeros so it is exactly 12 characters long
df_long['AREA'] = df_long['AREA'].astype(str).str.zfill(12)

Filter the date:

In [25]:
# 4. LOAD AND CONVERT THE SHAPEFILE

# Read the shapefile
gdf = gpd.read_file('/Users/lucasselvik/Desktop/ARC Project/tl_2020_06_bg/tl_2020_06_bg.shp')

# Ensure the shapefile ID is a string so it can match the CSV
# *Change 'GEOID' if your shapefile uses a different column name for the FIPS code
gdf['GEOID'] = gdf['GEOID'].astype(str)

Create Animated Heat Map:

In [26]:
# 5. Generate One Image Per Day
# We set a global min and max so the color scale means the exact same thing on Day 1 as it does on Day 31
vmin = 0

# Caps the darkest red at the 95th percentile, ignoring extreme outliers
vmax = df_long['Stops'].quantile(0.95)

# Loop through all 31 days of August
for day in range(1, 32):
    print(f"Generating map for August {day}...")
    
    # Isolate just this day's data
    daily_data = df_long[df_long['Day'] == day]
    
    # Merge the daily mobility data directly into the shapefile map
    merged = gdf.merge(daily_data, left_on='GEOID', right_on='AREA', how='left')
    
    # Setup the canvas for the image
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # Draw the map!
    merged.plot(
        column='Stops',        # What column determines the color
        cmap='OrRd',           # Color map (Orange-Red heat map)
        linewidth=0.1,         # Border thickness for the block groups
        ax=ax, 
        edgecolor='0.8',       # Border color (light grey)
        legend=True,           # Show the color bar scale
        vmin=vmin,             # Lock the minimum color
        vmax=vmax,             # Lock the maximum color
        missing_kwds={'color': 'lightgrey'} # Color for areas with no data
    )
    
    # Clean up the visual formatting
    ax.set_axis_off() # Removes the latitude/longitude box around the image
    ax.set_title(f"California Daily Mobility - August {day}, 2020", fontdict={'fontsize': 16, 'fontweight': 'bold'})
    
    # Save the image to the folder where your Jupyter Notebook lives
    filename = f"Mobility_August_{day:02d}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    
    # IMPORTANT: Close the figure so Jupyter doesn't run out of memory!
    plt.close(fig)

print("✅ Success! All 31 images have been generated and saved to your folder.")

Generating map for August 1...
Generating map for August 2...
Generating map for August 3...
Generating map for August 4...
Generating map for August 5...
Generating map for August 6...
Generating map for August 7...
Generating map for August 8...
Generating map for August 9...
Generating map for August 10...
Generating map for August 11...
Generating map for August 12...
Generating map for August 13...
Generating map for August 14...
Generating map for August 15...
Generating map for August 16...
Generating map for August 17...
Generating map for August 18...
Generating map for August 19...
Generating map for August 20...
Generating map for August 21...
Generating map for August 22...
Generating map for August 23...
Generating map for August 24...
Generating map for August 25...
Generating map for August 26...
Generating map for August 27...
Generating map for August 28...
Generating map for August 29...
Generating map for August 30...
Generating map for August 31...
✅ Success! All 31

Making a Video

In [28]:
import imageio
import os

# 1. Collect all the filenames we just created
image_folder = '.' # The current folder
images = []

# We create the list of filenames in exact order (01, 02, 03...)
filenames = [f"Mobility_August_{day:02d}.png" for day in range(1, 32)]

print("Reading images and building video...")

# 2. Read each image and add it to our list
for filename in filenames:
    if os.path.exists(filename):
        images.append(imageio.imread(filename))
    else:
        print(f"Warning: {filename} was not found and will be skipped.")

# 3. Save as an MP4 video
# fps=2 means 2 frames per second (the video will last about 15 seconds)
# You can increase fps to 5 or 10 if you want a faster "pulse" effect
video_filename = 'California_Mobility_August_2020.mp4'
imageio.mimsave(video_filename, images, fps=2)

print(f"✅ Success! Your video has been saved as: {video_filename}")

# 4. Optional: Play the video directly in your Jupyter Notebook
from IPython.display import Video
Video(video_filename, embed=True, width=700)

Reading images and building video...


/var/folders/l0/fdkh4qsd0zx5021lh7mjpd7w0000gn/T/ipykernel_49714/1049698371.py:16: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(filename))
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2457, 2472) to (2464, 2480) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[rawvideo @ 0x1396187a0] Stream #0: not enough frames to estimate rate; consider increasing probesize


✅ Success! Your video has been saved as: California_Mobility_August_2020.mp4
